In [1]:
from utils_sql import create_connection_to_vector_db, get_db_table

In [2]:
date_from = '2025-12-01'
date_to = '2025-12-10'

query_supplies = f"""SELECT supply_id, 
        product_id,
        SUM(quantity) AS tasks_count,
        DATE(created_at) AS date
    FROM inventory_transactions it
    WHERE DATE(created_at) BETWEEN '{date_from}'
    AND '{date_to}'
    AND it.delivery_type  = 'ФБС'
    GROUP BY supply_id, product_id, date;"""


# Устанавливаем соединение
connection = create_connection_to_vector_db()
# Делаем запрос
df_table_supllies = get_db_table(query_supplies, connection)

Соединение с БД PostgreSQL успешно установлено в 2025-12-15-11:M


c:\Users\123\Desktop\warehouse_service\research\utils_sql.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(db_query, connection).fillna(0).infer_objects(copy=False)


Данные из БД загружены в датафрейм


In [3]:
# Отбираем уникальные поставки
supllies_unique = df_table_supllies['supply_id'].unique()
# Достаем номера поставок
supllies_num = [suplly.removeprefix('WB-GI-') for suplly in supllies_unique]
# Количество СЗ в этих поставках
tasks_count = df_table_supllies['tasks_count'].sum()
print(f"Через сервис проведено {len(supllies_unique)} поставок за период с {date_from} по {date_to}.")
print()
print(f"В которых содржится {int(tasks_count)} сборочных заданий")

Через сервис проведено 975 поставок за период с 2025-12-01 по 2025-12-10.

В которых содржится 58487 сборочных заданий


In [4]:
# Вычислим сколько поставок в этот же период было по данным ВБ
query_supplies_by_wb = f"""SELECT 
	s.closed_at,
	s.id AS supply_id, 
	s."name",
	s2.id AS task_id,
	s2.local_vendor_code,
	s.account
FROM supplies_data s 
LEFT JOIN supplies_and_orders s2
ON s.id = s2.supply_id
WHERE DATE(s.closed_at) BETWEEN '{date_from}'
AND '{date_to}';"""
df_supplies_by_wb = get_db_table(query_supplies_by_wb, connection)
# Отберем уникальные поставки, по данным ВБ
supllies_unique_wb = df_supplies_by_wb['supply_id'].unique()

# Вычислим есть ли поставки на ВБ, которые не попали в наш сервис
supply_difference = set(supllies_unique_wb)-set(supllies_unique)
if len(supply_difference) > 0:
	print(f"Найдены следующие поставки, отсутствующие в сервисе:")
	print("\n".join(map(str, supply_difference)))

	# Проверим, что они не попали в inventory_transactions позже
	query_supplies_unfound = f"""SELECT supply_id, 
        product_id,
        SUM(quantity) AS tasks_count,
        DATE(created_at) AS date
    FROM inventory_transactions it
    WHERE it.delivery_type  = 'ФБС'
	AND supply_id IN {tuple(supply_difference)}
    GROUP BY supply_id, product_id, date;"""
	df_table_supllies_unfound = get_db_table(query_supplies_unfound, connection)
	count_tasks_conducted_later = df_table_supllies_unfound['tasks_count'].sum()
	supplies_conducted_later = df_table_supllies_unfound['supply_id'].unique()
	print(f"{count_tasks_conducted_later} СЗ проведены в сервисе позже, чем по данным ВБ")
	# Вычисляем окончательный перечень поставок, прошедших мимо сервиса
	supply_difference -= set(supplies_conducted_later)

	print()
	# Количество поставок, не попавших в сервис
	count_tasks_not_in_service = df_supplies_by_wb[df_supplies_by_wb['supply_id'].isin(supply_difference)]['task_id'].count() 
	print(f"Таким образом мимо сервиса прошло {count_tasks_not_in_service} сборочных заданий и {len(supply_difference)} поставок(ка)")
else:
	print("Расхождений не обнаружено")

Данные из БД загружены в датафрейм
Найдены следующие поставки, отсутствующие в сервисе:
WB-GI-199318780
WB-GI-201952302
WB-GI-201897750
WB-GI-201430422
WB-GI-201430420
WB-GI-201642145
WB-GI-200034651
WB-GI-200119918
WB-GI-201617182
WB-GI-201615806
WB-GI-201187312
WB-GI-200119914
WB-GI-200034652
WB-GI-201607614
WB-GI-201642137
WB-GI-201683229
WB-GI-201683228
WB-GI-201607611
WB-GI-201361690
WB-GI-201726175
WB-GI-200595518
WB-GI-200010264
WB-GI-201897753
WB-GI-201683227
WB-GI-201615807
WB-GI-200596139
WB-GI-201642139
WB-GI-200595520
WB-GI-201430421
WB-GI-201615805
WB-GI-201361692
WB-GI-199318779
WB-GI-201617183
WB-GI-200595517
WB-GI-201361694
WB-GI-201361691
WB-GI-201642151
WB-GI-200946770
WB-GI-201617180
WB-GI-200119913
WB-GI-201050136
WB-GI-201187313
WB-GI-201683226
WB-GI-200595515
WB-GI-201607610
WB-GI-201726180
WB-GI-201416453
WB-GI-201276624
WB-GI-201642148
WB-GI-201607609
WB-GI-200119920
WB-GI-200010263
WB-GI-200119915
WB-GI-200595519
WB-GI-201050135
WB-GI-201897752
WB-GI-200119916


In [5]:
supply_difference

{'WB-GI-199318779',
 'WB-GI-199318780',
 'WB-GI-200010263',
 'WB-GI-200010264',
 'WB-GI-200034651',
 'WB-GI-200034652',
 'WB-GI-200119913',
 'WB-GI-200119914',
 'WB-GI-200119915',
 'WB-GI-200119916',
 'WB-GI-200119917',
 'WB-GI-200119918',
 'WB-GI-200119920',
 'WB-GI-200595515',
 'WB-GI-200595517',
 'WB-GI-200595518',
 'WB-GI-200595519',
 'WB-GI-200595520',
 'WB-GI-200596139',
 'WB-GI-201050135',
 'WB-GI-201050136',
 'WB-GI-201276624',
 'WB-GI-201416453',
 'WB-GI-201607609',
 'WB-GI-201607610',
 'WB-GI-201607611',
 'WB-GI-201607613',
 'WB-GI-201607614',
 'WB-GI-201662270'}